# Phase 2 — Train RQ-VAE trên Kaggle

Notebook clone repository, tự tìm embedding trên Kaggle rồi chạy `train_rqvae.py`. Hyperparameter lấy từ `configs/rqvae_vmarket.gin`; notebook chỉ thay đường dẫn input.

Đầu vào bắt buộc từ notebook 02:

- `global_product_embeddings.f16.npy`
- `global_embedding_index.parquet`

Đầu ra chính:

- các checkpoint RQ-VAE;
- `semantic_ids.parquet` với ba cột `sid_0`, `sid_1`, `sid_2`;
- `semantic_id_metrics.json`.

Codebook được cố định ở `[128, 64, 32]`; không thêm collision suffix.

## Trước khi chạy

1. Bật GPU trong Kaggle Notebook Settings.
2. Add Data chứa output của notebook 02.
3. Tạo Kaggle Secret `GITHUB_TOKEN` có quyền đọc repository.
4. Tạo Kaggle Secret `WANDB_API_KEY`.
5. Notebook tự tìm embedding trong `/kaggle/input`; các cấu hình train còn lại lấy từ Gin.

## 0. Cấu hình

In [ ]:
from pathlib import Path

EMBEDDING_ROOT = None

GITHUB_REPOSITORY_URL = "https://github.com/nam-htran/VSF-MiniApp-Ecommerce.git"
GITHUB_BRANCH = "main"
REPOSITORY_ROOT = Path("/kaggle/working/vsf-miniapp-ecommerce-source")
AUTO_INSTALL_DEPENDENCIES = True

print("Configuration loaded.")

## 1. Cài dependency và kiểm tra GPU

In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys


required_modules = {
    "gin": "gin-config==0.5.0",
    "accelerate": "accelerate>=1.0.0",
    "einops": "einops>=0.8.0",
    "huggingface_hub": "huggingface-hub>=0.25.0",
    "wandb": "wandb>=0.19.0",
    "pyarrow": "pyarrow>=16.0.0",
}
missing_packages = [
    package
    for module, package in required_modules.items()
    if importlib.util.find_spec(module) is None
]
if AUTO_INSTALL_DEPENDENCIES and missing_packages:
    print("Installing:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

import pandas as pd
import pyarrow.parquet as pq
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {properties.name}, {properties.total_memory / 2**30:.1f} GiB")

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training RQ-VAE.")

## 2. Kết nối Weights & Biases

Đọc `WANDB_API_KEY` từ Kaggle Secret và đăng nhập W&B.

In [ ]:
import os


from kaggle_secrets import UserSecretsClient

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## 3. Clone source từ GitHub

Đọc `GITHUB_TOKEN` từ Kaggle Secret và clone nhánh `main`.

In [ ]:
from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPOSITORY_ROOT = Path(REPOSITORY_ROOT).expanduser().resolve()

git_environment = {
    **os.environ,
    "GITHUB_TOKEN": github_token,
    "GIT_TERMINAL_PROMPT": "0",
}
credential_helper = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
git = ["git", "-c", f"credential.helper={credential_helper}"]

if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run(
        [*git, "-C", str(REPOSITORY_ROOT), "pull", "--ff-only", "origin", GITHUB_BRANCH],
        check=True,
        env=git_environment,
    )
elif REPOSITORY_ROOT.exists():
    raise FileExistsError(f"Clone target is not a Git repository: {REPOSITORY_ROOT}")
else:
    subprocess.run(
        [*git, "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPOSITORY_URL, str(REPOSITORY_ROOT)],
        check=True,
        env=git_environment,
    )
del github_token, git_environment

SOURCE_ROOT = REPOSITORY_ROOT / "ai-recommendation/src"
if not (SOURCE_ROOT / "train_rqvae.py").is_file():
    raise FileNotFoundError(f"RQ-VAE source not found: {SOURCE_ROOT}")
print("SOURCE_ROOT:", SOURCE_ROOT)

## 4. Tìm embedding và tạo Gin runtime

In [ ]:
import json


def is_embedding_root(path):
    path = Path(path)
    return (
        (path / "global_product_embeddings.f16.npy").is_file()
        and (path / "global_embedding_index.parquet").is_file()
    )


def locate_embedding_root(explicit=None):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if is_embedding_root(root):
            return root
        raise FileNotFoundError(f"Notebook 02 artifacts were not found in {root}")

    cwd = Path.cwd().resolve()
    candidates = [
        Path("/kaggle/working/embeddings"),
        cwd / "embeddings",
        cwd.parent / "embeddings",
    ]
    for candidate in candidates:
        if is_embedding_root(candidate):
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for manifest_path in kaggle_input.glob("**/embedding_manifest.json"):
            if is_embedding_root(manifest_path.parent):
                return manifest_path.parent.resolve()
        for embedding_path in kaggle_input.glob("**/global_product_embeddings.f16.npy"):
            if is_embedding_root(embedding_path.parent):
                return embedding_path.parent.resolve()
    raise FileNotFoundError(
        "Notebook 02 output was not found. Add it as a Kaggle Dataset or set EMBEDDING_ROOT."
    )


EMBEDDING_ROOT = locate_embedding_root(EMBEDDING_ROOT)

BASE_CONFIG_PATH = SOURCE_ROOT / "configs/rqvae_vmarket.gin"
CONFIG_PATH = Path("/kaggle/working/rqvae_vmarket.gin")
config_lines = BASE_CONFIG_PATH.read_text(encoding="utf-8").splitlines()
binding = "train.dataset_folder="
matching_lines = [index for index, line in enumerate(config_lines) if line.startswith(binding)]
if len(matching_lines) != 1:
    raise ValueError(f"Expected one {binding} binding, found {len(matching_lines)}")
config_lines[matching_lines[0]] = binding + json.dumps(str(EMBEDDING_ROOT))
CONFIG_PATH.write_text("\n".join(config_lines) + "\n", encoding="utf-8")

print("Embedding root:", EMBEDDING_ROOT)
print("Gin config:", CONFIG_PATH)

## 5. Train RQ-VAE

In [ ]:
command = [sys.executable, "train_rqvae.py", str(CONFIG_PATH)]
print("Running:", " ".join(command))
subprocess.run(command, cwd=SOURCE_ROOT, check=True)

## 6. Kiểm tra artifacts

In [ ]:
OUTPUT_ROOT = Path("/kaggle/working/vmarket_rqvae")
semantic_ids_path = OUTPUT_ROOT / "semantic_ids.parquet"
metrics_path = OUTPUT_ROOT / "semantic_id_metrics.json"

In [ ]:
if not semantic_ids_path.is_file() or not metrics_path.is_file():
    raise FileNotFoundError("RQ-VAE finished without the required final artifacts.")

semantic_file = pq.ParquetFile(semantic_ids_path)
expected_columns = ["product_index", "product_id", "sid_0", "sid_1", "sid_2"]
if semantic_file.schema_arrow.names != expected_columns:
    raise ValueError(f"Unexpected semantic ID schema: {semantic_file.schema_arrow.names}")

metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
checkpoints = sorted(OUTPUT_ROOT.glob("checkpoint_*.pt"))
output_size = sum(path.stat().st_size for path in OUTPUT_ROOT.rglob("*") if path.is_file())

print("Final artifact validation: PASSED")
print("Semantic ID rows:", f"{semantic_file.metadata.num_rows:,}")
print("Checkpoints:", len(checkpoints))
print("Output size:", f"{output_size / 2**30:.2f} GiB")
print("OUTPUT_ROOT:", OUTPUT_ROOT)
display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))

## Khi nào notebook hoàn thành?

Notebook hoàn thành khi cell cuối báo `Final artifact validation: PASSED`. Sau đó lưu một Kaggle Notebook Version để giữ toàn bộ thư mục `/kaggle/working/vmarket_rqvae` làm output cho bước phân tích cluster và huấn luyện Transformer tiếp theo.